In [1]:
# ============================================================
# NB_INGST_BronzeSFTP_Person
# Capa: Bronze | Origen: SFTP | Tabla: Person
# Carga TOTAL — reemplaza todo en cada ejecución
# Todo desde config — sin hardcodeos (cumple TP Final)
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from datetime import datetime
import re

# ============================================================
# IDs de workspace y lakehouses — sin hardcodear (TP)
# El pipeline setea lh_landing como defaultLakehouse de este notebook.
# ============================================================
# lakehouse.get(nombre) resuelve por nombre desde el workspace.
# Funciona en ejecución manual y desde pipeline. WS_ID nunca hardcodeado.
_lh_landing = notebookutils.lakehouse.get('lh_landing')
WS_ID       = _lh_landing.workspaceId
ART_LANDING = _lh_landing.id
# ART_BRONZE no necesario: Bronze escribe con saveAsTable (nombre lógico)

# Leer config desde lh_config — nunca hardcodear parámetros
config = spark.sql("""
    SELECT *
    FROM lh_config.dbo.origin_bronze_silver
    WHERE origen       = 'SFTP'
      AND nombre_tabla = 'Person'
      AND activo       = 1
    LIMIT 1
""").collect()[0]

nombre_tabla  = config['nombre_tabla']          # Person
origen        = config['origen']                # SFTP
path_relativo = config['ubicacion_relativa']    # Files/TP_AdventureWorks/Person
tipo_carga    = config['tipo_carga']            # total
pks           = config['pks'].split(',') if config['pks'] else []
separador_csv = config['separador_csv'] if config['separador_csv'] else ','

# Path ABFS construido dinámicamente — sin hardcodear rutas (TP)
# Fabric requiere que el path empiece con 'Files/' para acceder a archivos.
# Se normaliza desde config — sin hardcodear la ruta.
_path = path_relativo.strip('/')
if not _path.startswith('Files/'):
    _path = 'Files/' + _path
abfs_origen = (
    f'abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com'
    f'/{ART_LANDING}/{_path}'
)

print('✅ Config cargada')
print(f'   nombre_tabla  : {nombre_tabla}')
print(f'   origen        : {origen}')
print(f'   tipo_carga    : {tipo_carga}')
print(f'   path_relativo : {path_relativo}')
print(f'   separador_csv : {separador_csv}')
print(f'   abfs_origen   : {abfs_origen}')


StatementMeta(, 01350aa8-df6e-46d8-bd66-e4dcce21517c, 3, Finished, Available, Finished, False)

✅ Config cargada
   nombre_tabla  : Person
   origen        : SFTP
   tipo_carga    : total
   path_relativo : TP_AdventureWorks/Person
   separador_csv : ;
   abfs_origen   : abfss://None@onelake.dfs.fabric.microsoft.com/f27e2853-1dca-44d9-94f8-08477a5836ff/TP_AdventureWorks/Person


In [2]:
# ============================================================
# Listar archivos en origen
# Carga total → procesar TODOS los CSV disponibles
# ============================================================

archivos_disponibles = notebookutils.fs.ls(abfs_origen)

archivos_todos = []
for f in archivos_disponibles:
    if f.name.endswith('.csv'):
        match = re.search(r'_(\d{8})\.csv$', f.name)
        fecha = match.group(1) if match else '00000000'
        archivos_todos.append({
            'path'  : f.path,
            'name'  : f.name,
            'fecha' : fecha
        })

print(f'📄 Archivos a procesar: {len(archivos_todos)}')
for a in archivos_todos:
    print(f'   → {a["name"]} (fecha: {a["fecha"]})')

if not archivos_todos:
    print('⚠️  Sin archivos CSV en origen')
    notebookutils.notebook.exit('sin_archivos')


StatementMeta(, 01350aa8-df6e-46d8-bd66-e4dcce21517c, 4, Finished, Available, Finished, False)

Py4JJavaError: An error occurred while calling z:notebookutils.fs.ls.
: Operation failed: "Bad Request", 400, GET, http://onelake.dfs.fabric.microsoft.com/None?upn=false&resource=filesystem&maxResults=5000&directory=f27e2853-1dca-44d9-94f8-08477a5836ff/TP_AdventureWorks/Person&timeout=90&recursive=false, FriendlyNameSupportDisabled, "Request Failed with WorkspaceId and ArtifactId should be either valid Guids or valid Names"
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.completeExecute(AbfsRestOperation.java:231)
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.lambda$execute$0(AbfsRestOperation.java:191)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDurationOfInvocation(IOStatisticsBinding.java:464)
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.execute(AbfsRestOperation.java:189)
	at org.apache.hadoop.fs.azurebfs.services.AbfsClient.listPath(AbfsClient.java:311)
	at org.apache.hadoop.fs.azurebfs.AzureBlobFileSystemStore.list

In [3]:
# ============================================================
# Carga total → borrar tabla anterior y recargar todo
# Castear TODO a string (requisito Bronze TP)
# LIMIT 1000 obligatorio durante desarrollo (TP)
# FIX: separador desde config — no hardcodeado
# FIX: DROP TABLE antes del loop para evitar schema conflict
# ============================================================

fecha_carga   = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
tabla_destino = f'lh_bronze.{origen}.{nombre_tabla}'
total_filas   = 0

# Carga total: eliminar tabla anterior (metastore + archivos)
spark.sql(f'DROP TABLE IF EXISTS {tabla_destino}')
print(f'🗑️  Tabla anterior eliminada: {tabla_destino}')

for archivo in archivos_todos:
    print(f'\n⚙️  Procesando: {archivo["name"]}')

    df_raw = (
        spark.read
        .option('header', 'true')
        .option('inferSchema', 'false')
        # FIX: separador desde config — nunca hardcodeado
        .option('sep', separador_csv)
        .csv(archivo['path'])
        .limit(1000)   # ← LIMIT 1000 obligatorio TP
    )

    print(f'   Columnas raw : {len(df_raw.columns)}')
    print(f'   Filas raw    : {df_raw.count()}')

    # Limpiar nombres de columnas + castear TODO a string (Bronze)
    df_bronze = df_raw.select([
        F.col(c).cast(StringType())
         .alias(re.sub(r'[ ,;{}()\n\t=]', '_', c.strip()))
        for c in df_raw.columns
    ])

    # Columnas de auditoría (obligatorias en Bronze según TP)
    df_bronze = (
        df_bronze
        .withColumn('fecha_carga',    F.lit(fecha_carga))
        .withColumn('nombre_archivo', F.lit(archivo['name']))
        .withColumn('fecha_archivo',  F.lit(archivo['fecha']))
    )

    (
        df_bronze.write
        .format('delta')
        .mode('append')
        .option('mergeSchema', 'true')
        .saveAsTable(tabla_destino)
    )

    filas = df_bronze.count()
    total_filas += filas
    print(f'   ✅ {filas} filas → {tabla_destino}')

print(f'\n✅ TOTAL filas cargadas : {total_filas}')
print(f'✅ Tabla destino        : {tabla_destino}')


StatementMeta(, 01350aa8-df6e-46d8-bd66-e4dcce21517c, 5, Finished, Available, Finished, True)

🗑️  Tabla anterior eliminada: lh_bronze.SFTP.Person


NameError: name 'archivos_todos' is not defined

In [4]:
# ============================================================
# Verificar resultado en Bronze
# ============================================================

display(spark.sql(f"""
    SELECT
        nombre_archivo,
        fecha_archivo,
        fecha_carga,
        COUNT(*) AS cant_filas
    FROM {tabla_destino}
    GROUP BY nombre_archivo, fecha_archivo, fecha_carga
    ORDER BY fecha_archivo
"""))

total = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabla_destino}').collect()[0]['cnt']
print(f'✅ Total filas en Bronze ({tabla_destino}): {total}')


StatementMeta(, 01350aa8-df6e-46d8-bd66-e4dcce21517c, 6, Finished, Available, Finished, True)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `lh_bronze`.`SFTP`.`Person` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 7 pos 9;
'Sort ['fecha_archivo ASC NULLS FIRST], true
+- 'Aggregate ['nombre_archivo, 'fecha_archivo, 'fecha_carga], ['nombre_archivo, 'fecha_archivo, 'fecha_carga, count(1) AS cant_filas#1281L]
   +- 'UnresolvedRelation [lh_bronze, SFTP, Person], [], false
